# Hidden strength features (ATP)

Цель: проверить, дают ли дополнительные признаки «скрытой силы» игрока (нагрузка, серии, профиль по покрытию, матчап по руке) улучшение качества, особенно в режиме **no-odds**.

In [1]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score

os.makedirs('figures', exist_ok=True)

ods = pd.read_parquet('ods.parquet')
tml = pd.read_parquet('tml.parquet')
players = pd.read_parquet('players.parquet')

ods['Date'] = pd.to_datetime(ods['Date'])
tml['tourney_date'] = pd.to_datetime(tml['tourney_date'].astype(str), format='%Y%m%d', errors='coerce')

(ods.shape, tml.shape, players.shape)

((22851, 42), (24939, 50), (7643, 12))

In [2]:
def key_from_odds(name: str) -> str:
    if pd.isna(name):
        return ''
    s = str(name).strip().lower()
    s = s.replace("'", "").replace('-', ' ')
    s = re.sub(r"\s+", " ", s)
    parts = s.split(' ')
    if not parts:
        return ''
    init = re.sub(r"[^a-z]", "", parts[-1].replace('.', ''))[:1]
    surname = re.sub(r"[^a-z\s]", "", ' '.join(parts[:-1])).strip()
    return f"{surname} {init}".strip()


def key_from_tml(full_name: str) -> str:
    if pd.isna(full_name):
        return ''
    s = str(full_name).strip().lower()
    s = s.replace("'", "").replace('-', ' ')
    s = re.sub(r"\s+", " ", s)
    parts = s.split(' ')
    if not parts:
        return ''
    if len(parts) == 1:
        return re.sub(r"[^a-z]", "", parts[0])
    init = re.sub(r"[^a-z]", "", parts[0])[:1]
    surname = re.sub(r"[^a-z\s]", "", ' '.join(parts[1:])).strip()
    return f"{surname} {init}".strip()


ods['week'] = (ods['Date'] - pd.to_timedelta(ods['Date'].dt.weekday, unit='D')).dt.normalize()
tml['week'] = tml['tourney_date'].dt.normalize()

ods['wkey'] = ods['Winner'].map(key_from_odds)
ods['lkey'] = ods['Loser'].map(key_from_odds)
tml['wkey'] = tml['winner_name'].map(key_from_tml)
tml['lkey'] = tml['loser_name'].map(key_from_tml)

ods['p1'] = ods[['wkey', 'lkey']].min(axis=1)
ods['p2'] = ods[['wkey', 'lkey']].max(axis=1)
tml['p1'] = tml[['wkey', 'lkey']].min(axis=1)
tml['p2'] = tml[['wkey', 'lkey']].max(axis=1)

joined = tml.merge(
    ods,
    how='inner',
    on=['week', 'p1', 'p2'],
    suffixes=('_tml', '_ods')
)

print('joined shape:', joined.shape)
print('match rate tml:', round(joined.shape[0] / tml.shape[0], 3))
print('match rate ods:', round(joined.shape[0] / ods.shape[0], 3))
print('surface agree:', round((joined['surface'].str.lower() == joined['Surface'].str.lower()).mean(), 4))

joined[['week','winner_name','loser_name','Winner','Loser','surface','Surface','B365W','B365L','PSW','PSL']].head(3)

joined shape: (17633, 99)
match rate tml: 0.707
match rate ods: 0.772
surface agree: 0.996


,week,winner_name,loser_name,Winner,Loser,surface,Surface,B365W,B365L,PSW,PSL
0,2017-01-02,Viktor Troicki,Yoshihito Nishioka,Troicki V.,Nishioka Y.,Hard,Hard,1.50,2.5,1.54,2.62
1,2017-01-02,Kyle Edmund,Ernesto Escobedo,Edmund K.,Escobedo E.,Hard,Hard,1.36,3.0,1.42,3.09
2,2017-01-02,Lucas Pouille,Gilles Simon,Pouille L.,Simon G.,Hard,Hard,1.80,1.9,1.93,1.95


In [3]:
# Build symmetric match frame: target is whether p1 won
joined = joined.copy()

joined['winner_key'] = joined['winner_name'].map(key_from_tml)
joined['loser_key'] = joined['loser_name'].map(key_from_tml)

joined['y_p1_win'] = (joined['winner_key'] == joined['p1']).astype(int)

# ranks for p1/p2
joined['rank_p1'] = np.where(joined['winner_key'] == joined['p1'], joined['winner_rank'], joined['loser_rank'])
joined['rank_p2'] = np.where(joined['winner_key'] == joined['p2'], joined['winner_rank'], joined['loser_rank'])
joined['rank_diff_p1_minus_p2'] = joined['rank_p1'] - joined['rank_p2']

joined['rank_points_p1'] = np.where(joined['winner_key'] == joined['p1'], joined['winner_rank_points'], joined['loser_rank_points'])
joined['rank_points_p2'] = np.where(joined['winner_key'] == joined['p2'], joined['winner_rank_points'], joined['loser_rank_points'])
joined['rank_points_diff_p1_minus_p2'] = joined['rank_points_p1'] - joined['rank_points_p2']

# odds for p1/p2 (B365 and Pinnacle PS)
joined['b365_odds_p1'] = np.where(joined['winner_key'] == joined['p1'], joined['B365W'], joined['B365L'])
joined['b365_odds_p2'] = np.where(joined['winner_key'] == joined['p2'], joined['B365W'], joined['B365L'])

joined['ps_odds_p1'] = np.where(joined['winner_key'] == joined['p1'], joined['PSW'], joined['PSL'])
joined['ps_odds_p2'] = np.where(joined['winner_key'] == joined['p2'], joined['PSW'], joined['PSL'])

# implied probabilities (normalize overround per match)
for prefix in ['b365', 'ps']:
    op1 = joined[f'{prefix}_odds_p1']
    op2 = joined[f'{prefix}_odds_p2']
    p1 = 1.0 / op1
    p2 = 1.0 / op2
    s = p1 + p2
    joined[f'{prefix}_imp_p1'] = (p1 / s).where(np.isfinite(s), np.nan)
    joined[f'{prefix}_imp_p2'] = (p2 / s).where(np.isfinite(s), np.nan)

# hands: derive hand for p1/p2 and opponent hand
joined['hand_p1'] = np.where(joined['winner_key'] == joined['p1'], joined['winner_hand'], joined['loser_hand'])
joined['hand_p2'] = np.where(joined['winner_key'] == joined['p2'], joined['winner_hand'], joined['loser_hand'])
joined['opp_hand_for_p1'] = joined['hand_p2']
joined['opp_hand_for_p2'] = joined['hand_p1']

# ensure match date exists
joined['match_date'] = joined['tourney_date']

joined[['match_date','p1','p2','y_p1_win','ps_imp_p1','b365_imp_p1','rank_diff_p1_minus_p2','surface','hand_p1','hand_p2']].head(3)

,match_date,p1,p2,y_p1_win,ps_imp_p1,b365_imp_p1,rank_diff_p1_minus_p2,surface,hand_p1,hand_p2
0,2017-01-02,nishioka y,troicki v,0,0.370192,0.375000,71.0,Hard,L,R
1,2017-01-02,edmund k,escobedo e,1,0.685144,0.688073,-96.0,Hard,R,R
2,2017-01-02,pouille l,simon g,1,0.502577,0.513514,-10.0,Hard,R,R


In [4]:
def add_h2h_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['match_date', 'tourney_id', 'match_num']).copy()

    # key independent of p1/p2 order
    df['pair_key'] = df['p1'] + '|' + df['p2']

    # winner relative to p1
    df['p1_win'] = df['y_p1_win'].astype(int)

    # cumulative h2h before current match
    df['h2h_p1_wins_before'] = df.groupby('pair_key')['p1_win'].cumsum().shift(1).fillna(0).astype(int)
    df['h2h_matches_before'] = df.groupby('pair_key').cumcount()

    # centered feature: (p1 wins - p2 wins)
    df['h2h_p1_minus_p2_before'] = df['h2h_p1_wins_before'] - (df['h2h_matches_before'] - df['h2h_p1_wins_before'])

    return df.drop(columns=['pair_key', 'p1_win'])


def add_form_features(df: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    df = df.sort_values(['match_date', 'tourney_id', 'match_num']).copy()
    df = df.reset_index(drop=True)

    df['match_row_id'] = np.arange(len(df), dtype=int)

    long = pd.concat([
        pd.DataFrame({
            'match_row_id': df['match_row_id'],
            'match_date': df['match_date'],
            'role': 'p1',
            'player': df['p1'],
            'is_win': df['y_p1_win'].astype(int),
        }),
        pd.DataFrame({
            'match_row_id': df['match_row_id'],
            'match_date': df['match_date'],
            'role': 'p2',
            'player': df['p2'],
            'is_win': (1 - df['y_p1_win'].astype(int)),
        }),
    ], ignore_index=True)

    long = long.sort_values(['player', 'match_date', 'match_row_id'])

    long['form_winrate'] = (
        long.groupby('player')['is_win']
        .transform(lambda s: s.shift(1).rolling(n, min_periods=1).mean())
    )

    wide = (
        long.pivot(index='match_row_id', columns='role', values='form_winrate')
        .rename(columns={'p1': f'form_p1_last{n}', 'p2': f'form_p2_last{n}'})
        .reset_index()
    )

    out = df.merge(wide, on='match_row_id', how='left')
    out[f'form_diff_p1_minus_p2_last{n}'] = out[f'form_p1_last{n}'] - out[f'form_p2_last{n}']

    return out.drop(columns=['match_row_id'])


def _streaks_before(is_win: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    win_streak_before = np.zeros(len(is_win), dtype=int)
    lose_streak_before = np.zeros(len(is_win), dtype=int)

    w = 0
    l = 0
    for i, v in enumerate(is_win):
        win_streak_before[i] = w
        lose_streak_before[i] = l
        if v == 1:
            w += 1
            l = 0
        else:
            l += 1
            w = 0

    return win_streak_before, lose_streak_before


def add_hidden_strength_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(['match_date', 'tourney_id', 'match_num']).reset_index(drop=True).copy()
    out['match_row_id'] = np.arange(len(out), dtype=int)

    long = pd.concat([
        pd.DataFrame({
            'match_row_id': out['match_row_id'],
            'match_date': out['match_date'],
            'surface': out['surface'],
            'player': out['p1'],
            'role': 'p1',
            'is_win': out['y_p1_win'].astype(int),
            'player_hand': out['hand_p1'],
            'opp_hand': out['opp_hand_for_p1'],
        }),
        pd.DataFrame({
            'match_row_id': out['match_row_id'],
            'match_date': out['match_date'],
            'surface': out['surface'],
            'player': out['p2'],
            'role': 'p2',
            'is_win': (1 - out['y_p1_win'].astype(int)),
            'player_hand': out['hand_p2'],
            'opp_hand': out['opp_hand_for_p2'],
        }),
    ], ignore_index=True)

    long = long.sort_values(['player', 'match_date', 'match_row_id']).reset_index(drop=True)

    # Load / match frequency features
    # Important: match_date can have duplicates (multiple matches same day). We create a strictly increasing
    # per-player timestamp to allow rolling windows. The returned Series must keep g.index to align on assignment.
    def _count_prev_in_window(g: pd.DataFrame, window: str) -> pd.Series:
        g = g.sort_values(['match_date', 'match_row_id']).copy()
        ts = g['match_date'] + pd.to_timedelta(np.arange(len(g), dtype=np.int64), unit='ns')
        s = pd.Series(1, index=ts)
        values = s.rolling(window, closed='left').sum().to_numpy()
        return pd.Series(values, index=g.index)

    long['matches_last7d'] = (
        long.groupby('player', group_keys=False)
        .apply(lambda g: _count_prev_in_window(g, '7D'))
        .sort_index()
    )
    long['matches_last30d'] = (
        long.groupby('player', group_keys=False)
        .apply(lambda g: _count_prev_in_window(g, '30D'))
        .sort_index()
    )

    # Win/lose streaks
    def _add_streaks(g: pd.DataFrame) -> pd.DataFrame:
        w, l = _streaks_before(g['is_win'].to_numpy(dtype=int))
        g = g.copy()
        g['win_streak_before'] = w
        g['lose_streak_before'] = l
        return g

    long = long.groupby('player', group_keys=False).apply(_add_streaks)

    # Surface-specific preference (winrate before)
    def _expanding_mean_before(s: pd.Series) -> pd.Series:
        return s.shift(1).expanding(min_periods=1).mean()

    long['surface_winrate_before'] = long.groupby(['player', 'surface'])['is_win'].transform(_expanding_mean_before)
    long['overall_winrate_before'] = long.groupby('player')['is_win'].transform(_expanding_mean_before)

    # Matchup vs opponent hand (L/R)
    long['opp_hand_norm'] = long['opp_hand'].astype(str).str.upper().replace({'NAN': np.nan, '': np.nan})
    long.loc[~long['opp_hand_norm'].isin(['L', 'R']), 'opp_hand_norm'] = np.nan

    long['winrate_vs_opp_hand_before'] = long.groupby(['player', 'opp_hand_norm'])['is_win'].transform(_expanding_mean_before)

    # Player hand itself as a feature (will be used as categorical)
    long['player_hand_norm'] = long['player_hand'].astype(str).str.upper().replace({'NAN': np.nan, '': np.nan})
    long.loc[~long['player_hand_norm'].isin(['L', 'R']), 'player_hand_norm'] = np.nan

    # Pivot back to match-level
    wide_num = (
        long.pivot(index='match_row_id', columns='role', values=[
            'matches_last7d',
            'matches_last30d',
            'win_streak_before',
            'lose_streak_before',
            'surface_winrate_before',
            'overall_winrate_before',
            'winrate_vs_opp_hand_before',
        ])
    )
    wide_num.columns = [f'{k}_{role}' for k, role in wide_num.columns]
    wide_num = wide_num.reset_index()

    wide_cat = (
        long.pivot(index='match_row_id', columns='role', values=[
            'player_hand_norm',
            'opp_hand_norm',
        ])
    )
    wide_cat.columns = [f'{k}_{role}' for k, role in wide_cat.columns]
    wide_cat = wide_cat.reset_index()

    out = out.merge(wide_num, on='match_row_id', how='left').merge(wide_cat, on='match_row_id', how='left')

    # Differences p1 - p2
    diff_pairs = [
        ('matches_last7d', 'load_diff_last7d'),
        ('matches_last30d', 'load_diff_last30d'),
        ('win_streak_before', 'win_streak_diff_before'),
        ('lose_streak_before', 'lose_streak_diff_before'),
        ('surface_winrate_before', 'surface_winrate_diff_before'),
        ('overall_winrate_before', 'overall_winrate_diff_before'),
        ('winrate_vs_opp_hand_before', 'winrate_vs_opp_hand_diff_before'),
    ]

    for base, out_name in diff_pairs:
        out[out_name] = out[f'{base}_p1'] - out[f'{base}_p2']

    return out


joined_h = add_h2h_features(joined)
joined_h = add_form_features(joined_h, n=10)
joined_h = add_hidden_strength_features(joined_h)

cols_preview = [
    'match_date', 'p1', 'p2', 'y_p1_win', 'surface',
    'h2h_p1_minus_p2_before', 'form_diff_p1_minus_p2_last10',
    'load_diff_last7d', 'load_diff_last30d',
    'win_streak_diff_before', 'overall_winrate_diff_before',
    'surface_winrate_diff_before', 'winrate_vs_opp_hand_diff_before',
    'player_hand_norm_p1', 'player_hand_norm_p2',
    'opp_hand_norm_p1', 'opp_hand_norm_p2',
]
joined_h[cols_preview].head(5)

/tmp/ipykernel_319192/1184396677.py:122: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: _count_prev_in_window(g, '7D'))
/tmp/ipykernel_319192/1184396677.py:127: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: _count_prev_in_window(g, '30D'))
/tmp/ipykernel_319192/1184396677.py:139: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and i

,match_date,p1,p2,y_p1_win,surface,h2h_p1_minus_p2_before,form_diff_p1_minus_p2_last10,load_diff_last7d,load_diff_last30d,win_streak_diff_before,overall_winrate_diff_before,surface_winrate_diff_before,winrate_vs_opp_hand_diff_before,player_hand_norm_p1,player_hand_norm_p2,opp_hand_norm_p1,opp_hand_norm_p2
0,2017-01-02,nishioka y,troicki v,0,Hard,0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,L,R,R,L
1,2017-01-02,edmund k,escobedo e,1,Hard,0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,R,R,R,R
2,2017-01-02,pouille l,simon g,1,Hard,2,NaN,NaN,NaN,0.0,NaN,NaN,NaN,R,R,R,R
3,2017-01-02,donaldson j,muller g,1,Hard,2,NaN,NaN,NaN,0.0,NaN,NaN,NaN,R,L,L,R
4,2017-01-02,ferrer d,tomic b,1,Hard,2,NaN,NaN,NaN,0.0,NaN,NaN,NaN,R,R,R,R


In [ ]:
from sklearn.preprocessing import StandardScaler


def eval_probs(y_true, p_pred, name: str):
    y_true = np.asarray(y_true, dtype=float)
    p_pred = np.asarray(p_pred, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(p_pred)
    y_true = y_true[mask].astype(int)
    p_pred = p_pred[mask].clip(1e-6, 1 - 1e-6)

    return {
        'model': name,
        'n': int(len(y_true)),
        'logloss': float(log_loss(y_true, p_pred)),
        'roc_auc': float(roc_auc_score(y_true, p_pred)),
        'acc@0.5': float(accuracy_score(y_true, p_pred >= 0.5)),
    }


def run_models(df: pd.DataFrame, cutoff: str = '2024-01-01') -> pd.DataFrame:
    cutoff_ts = pd.Timestamp(cutoff)
    model_df = df.dropna(subset=['y_p1_win', 'week']).copy()

    train_df = model_df[model_df['week'] < cutoff_ts].copy()
    test_df = model_df[model_df['week'] >= cutoff_ts].copy()

    y_train = train_df['y_p1_win'].astype(int)
    y_test = test_df['y_p1_win'].astype(int)

    # Feature sets
    base_no_odds_num = [
        'rank_diff_p1_minus_p2',
        'rank_points_diff_p1_minus_p2',
        'h2h_p1_minus_p2_before',
        'form_diff_p1_minus_p2_last10',
    ]

    hidden_num = [
        'load_diff_last7d',
        'load_diff_last30d',
        'win_streak_diff_before',
        'lose_streak_diff_before',
        'overall_winrate_diff_before',
        'surface_winrate_diff_before',
        'winrate_vs_opp_hand_diff_before',
    ]

    odds_num = ['ps_imp_p1', 'b365_imp_p1']
    cat_cols = ['surface', 'player_hand_norm_p1', 'player_hand_norm_p2', 'opp_hand_norm_p1', 'opp_hand_norm_p2']

    def _existing(cols, frame):
        return [c for c in cols if c in frame.columns]

    sets = {
        'baseline_ps_only': ['ps_imp_p1'],

        # no-odds
        'no_odds_base': _existing(base_no_odds_num, train_df) + _existing(['surface'], train_df),
        'no_odds_base+hidden': _existing(base_no_odds_num, train_df) + _existing(hidden_num, train_df) + _existing(cat_cols, train_df),

        # with-odds
        'with_odds': _existing(odds_num, train_df),
        'with_odds+hidden': _existing(odds_num, train_df) + _existing(hidden_num, train_df) + _existing(cat_cols, train_df),
    }

    rows = []

    for name, cols in sets.items():
        cols = _existing(cols, train_df)
        if not cols:
            continue

        if name == 'baseline_ps_only':
            if 'ps_imp_p1' in test_df.columns:
                rows.append({**eval_probs(y_test, test_df['ps_imp_p1'], name), 'cutoff': cutoff})
            continue

        num_cols = [c for c in cols if c not in cat_cols]
        local_cat_cols = [c for c in cols if c in cat_cols]

        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            # with_mean=False to support sparse matrices after OneHot
            ('scaler', StandardScaler(with_mean=False)),
        ])
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ])

        pre = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, num_cols),
                ('cat', categorical_transformer, local_cat_cols),
            ],
            remainder='drop'
        )

        for model_name, model in [
            # saga обычно устойчивее на разреженных one-hot + mixed-scale числах
            ('logreg', LogisticRegression(
                max_iter=10000,
                solver='saga',
                penalty='l2',
                n_jobs=-1,
            )),
            ('gnb', GaussianNB()),
        ]:
            pipe = Pipeline(steps=[('preprocess', pre), ('model', model)])

            X_train = train_df[cols].copy()
            X_test = test_df[cols].copy()

            # Keep odds usable
            for oc in ['ps_imp_p1', 'b365_imp_p1']:
                if oc in X_train.columns:
                    X_train[oc] = X_train[oc].fillna(0.5)
                    X_test[oc] = X_test[oc].fillna(0.5)

            pipe.fit(X_train, y_train)
            p = pipe.predict_proba(X_test)[:, 1]

            rows.append({
                **eval_probs(y_test, p, f'{name}__{model_name}'),
                'cutoff': cutoff,
            })

    return pd.DataFrame(rows).sort_values('logloss')


res_2024 = run_models(joined_h, cutoff='2024-01-01')
res_2025 = run_models(joined_h, cutoff='2025-01-01')

all_hidden_res = pd.concat([res_2024, res_2025], ignore_index=True)
all_hidden_res.sort_values(['cutoff', 'logloss']).reset_index(drop=True)

/home/lex/learn/bfu/course_work/atp_tml_database/.venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/lex/learn/bfu/course_work/atp_tml_database/.venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as s

,model,n,logloss,roc_auc,acc@0.5,cutoff
0,baseline_ps_only,3448,0.598630,0.740271,0.675754,2024-01-01
1,with_odds__logreg,3535,0.599290,0.739859,0.676662,2024-01-01
2,with_odds+hidden__logreg,3535,0.599765,0.738909,0.676379,2024-01-01
3,no_odds_base+hidden__logreg,3535,0.631476,0.695876,0.639038,2024-01-01
4,no_odds_base__logreg,3535,0.633116,0.693914,0.639321,2024-01-01
5,no_odds_base__gnb,3535,0.657223,0.688611,0.636492,2024-01-01
6,with_odds__gnb,3535,0.668645,0.741366,0.676662,2024-01-01
7,no_odds_base+hidden__gnb,3535,0.842801,0.670964,0.619519,2024-01-01
8,with_odds+hidden__gnb,3535,0.901775,0.714046,0.652051,2024-01-01
9,with_odds+hidden__logreg,1702,0.607887,0.730276,0.668038,2025-01-01


## Weather features (optional)

Источник: Open-Meteo (геокодинг + исторические дневные значения).  
Особенности:

- используется **SQLite-кэш** (`weather_cache.sqlite`) в корне проекта;
- запросы выполняются только при отсутствии значения в кэше;
- погода добавляется только для матчей `Court == Outdoor` (поле берётся из `ods.parquet` после join);
- если `Location` отсутствует или геокодинг не сработал — значения остаются `NaN`.

Важно: эта часть делает внешние HTTP-запросы при выполнении ячеек.

In [6]:
import json
import sqlite3
from typing import Optional, Dict, Any

import requests

CACHE_DB_PATH = 'weather_cache.sqlite'


def _get_conn() -> sqlite3.Connection:
    conn = sqlite3.connect(CACHE_DB_PATH)
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS geocode_cache (
            location TEXT PRIMARY KEY,
            lat REAL,
            lon REAL,
            raw_json TEXT
        )
        """
    )
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS weather_cache (
            location TEXT,
            date TEXT,
            lat REAL,
            lon REAL,
            tmax REAL,
            tmin REAL,
            precip REAL,
            windmax REAL,
            raw_json TEXT,
            PRIMARY KEY (location, date)
        )
        """
    )
    conn.commit()
    return conn


def geocode_location(location: str) -> Optional[Dict[str, Any]]:
    if location is None:
        return None
    location = str(location).strip()
    if not location:
        return None

    conn = _get_conn()
    try:
        row = conn.execute(
            "SELECT lat, lon, raw_json FROM geocode_cache WHERE location = ?",
            (location,),
        ).fetchone()
        if row is not None:
            lat, lon, raw_json = row
            if lat is None or lon is None:
                return None
            return {'location': location, 'lat': float(lat), 'lon': float(lon), 'raw': json.loads(raw_json) if raw_json else None}

        url = 'https://geocoding-api.open-meteo.com/v1/search'
        params = {
            'name': location,
            'count': 1,
            'language': 'en',
            'format': 'json',
        }
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        results = data.get('results') or []
        if not results:
            conn.execute(
                "INSERT OR REPLACE INTO geocode_cache(location, lat, lon, raw_json) VALUES (?, ?, ?, ?)",
                (location, None, None, json.dumps(data)),
            )
            conn.commit()
            return None

        best = results[0]
        lat = float(best['latitude'])
        lon = float(best['longitude'])

        conn.execute(
            "INSERT OR REPLACE INTO geocode_cache(location, lat, lon, raw_json) VALUES (?, ?, ?, ?)",
            (location, lat, lon, json.dumps(best)),
        )
        conn.commit()

        return {'location': location, 'lat': lat, 'lon': lon, 'raw': best}
    finally:
        conn.close()


def fetch_daily_weather(location: str, date: pd.Timestamp) -> Optional[Dict[str, Any]]:
    if date is None or pd.isna(date):
        return None

    date_str = pd.Timestamp(date).date().isoformat()

    conn = _get_conn()
    try:
        row = conn.execute(
            """
            SELECT lat, lon, tmax, tmin, precip, windmax, raw_json
            FROM weather_cache
            WHERE location = ? AND date = ?
            """,
            (location, date_str),
        ).fetchone()
        if row is not None:
            lat, lon, tmax, tmin, precip, windmax, raw_json = row
            return {
                'location': location,
                'date': date_str,
                'lat': lat,
                'lon': lon,
                'tmax': tmax,
                'tmin': tmin,
                'precip': precip,
                'windmax': windmax,
                'raw': json.loads(raw_json) if raw_json else None,
            }

        geo = geocode_location(location)
        if geo is None:
            conn.execute(
                "INSERT OR REPLACE INTO weather_cache(location, date, lat, lon, tmax, tmin, precip, windmax, raw_json) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                (location, date_str, None, None, None, None, None, None, None),
            )
            conn.commit()
            return None

        lat = geo['lat']
        lon = geo['lon']

        url = 'https://archive-api.open-meteo.com/v1/archive'
        params = {
            'latitude': lat,
            'longitude': lon,
            'start_date': date_str,
            'end_date': date_str,
            'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max',
            'timezone': 'UTC',
        }

        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()

        daily = data.get('daily') or {}
        tmax = (daily.get('temperature_2m_max') or [None])[0]
        tmin = (daily.get('temperature_2m_min') or [None])[0]
        precip = (daily.get('precipitation_sum') or [None])[0]
        windmax = (daily.get('windspeed_10m_max') or [None])[0]

        conn.execute(
            "INSERT OR REPLACE INTO weather_cache(location, date, lat, lon, tmax, tmin, precip, windmax, raw_json) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
            (location, date_str, lat, lon, tmax, tmin, precip, windmax, json.dumps(data)),
        )
        conn.commit()

        return {
            'location': location,
            'date': date_str,
            'lat': lat,
            'lon': lon,
            'tmax': tmax,
            'tmin': tmin,
            'precip': precip,
            'windmax': windmax,
            'raw': data,
        }
    finally:
        conn.close()

In [7]:
def add_weather_features(df: pd.DataFrame, limit_rows: int = 500) -> pd.DataFrame:
    out = df.copy()

    # Open-Meteo: only for Outdoor matches and known location/date
    if 'Court' not in out.columns:
        raise ValueError("Column 'Court' not found. Expected it from odds (ods) after join.")
    if 'Location' not in out.columns:
        raise ValueError("Column 'Location' not found. Expected it from odds (ods) after join.")

    out['is_outdoor'] = out['Court'].astype(str).str.lower().eq('outdoor')

    # Initialize columns
    for c in ['wx_tmax', 'wx_tmin', 'wx_precip', 'wx_windmax']:
        if c not in out.columns:
            out[c] = np.nan

    mask = out['is_outdoor'] & out['Location'].notna() & out['match_date'].notna()
    idx = out.index[mask].tolist()

    # Protect yourself from long runs: by default only fetch for a subset
    if limit_rows is not None:
        idx = idx[: int(limit_rows)]

    for i in idx:
        loc = str(out.at[i, 'Location'])
        dt = pd.Timestamp(out.at[i, 'match_date'])
        try:
            w = fetch_daily_weather(loc, dt)
        except Exception:
            # Keep NaNs on errors
            continue
        if not w:
            continue

        out.at[i, 'wx_tmax'] = w.get('tmax')
        out.at[i, 'wx_tmin'] = w.get('tmin')
        out.at[i, 'wx_precip'] = w.get('precip')
        out.at[i, 'wx_windmax'] = w.get('windmax')

    return out


# Example usage (uncomment to run):
# joined_hw = add_weather_features(joined_h, limit_rows=500)
# joined_hw[['Location','match_date','Court','wx_tmax','wx_precip','wx_windmax']].head(10)